# B0 frozen-split retrain on Colab (session-proof)

Trains `SegResNetB0` on the official frozen split (`data/manifests/split_v1.json`) with the exact
recipe of `scripts/train_segmentation.py` (seed 17, 150 epochs, AdamW 3e-4, accum 2, Dice+0.5*CE,
96^3 patch, foreground prob 0.7, cosine schedule, clip-after-unscale) so B1/H0 stay comparable.

Design decisions baked in:
- data is copied once per session from Drive to the local SSD (Drive FUSE is too slow for per-epoch reads)
- checkpoints (`last.pt` / `best.pt`) live on Drive, so a disconnect loses at most one epoch
- only `train` + `val` IDs are ever read; `test`/`calibration` stay locked; split hash + commit land in `run.json`

One-time setup: put the training data on Drive as EITHER the extracted folder at
`CONFIG['drive_data_root']` OR a single .zip of the case folders at `CONFIG['drive_zip']` - the zip
uploads far more reliably than thousands of loose NIfTI files and is unzipped straight to local SSD
each session. Also make sure `split_v1.json` (+ `.sha256`) is either in the cloned repo or on Drive.

## Running this from VS Code

This file is a normal `.ipynb`: edit it in VS Code, then pick ONE of these execution routes.

**Route A (recommended): edit in VS Code, run on colab.research.google.com.** Push or upload the
file, open it at colab.research.google.com (File > Upload notebook), attach a GPU runtime
(Runtime > Change runtime type > T4 GPU), Run all. Results are identical; this is the least fragile path.

**Route B: true VS Code kernel on the Colab VM (SSH tunnel).** Start a GPU session in the browser
once, run the bootstrap below in a scratch cell there, then connect VS Code over SSH and select the
VM's python as the kernel:

```
# scratch cell in BROWSER colab, once per session:
!pip install -q colab_ssh
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="scanvidence")
# -> copy the printed VS Code Remote-SSH Host block into ~/.ssh/config,
#    then VS Code: Remote-SSH > Connect to Host, open /content/Scanvidence,
#    select kernel: Python (/usr/bin/python3).
```

After a reconnect (either route): re-run cells 1-7. Cell 4 skips already-copied cases and cell 6
reloads `last.pt` from Drive; worst case you lose one epoch.

In [ ]:
CONFIG = {
    "repo_url": "https://github.com/Scanvidence/Scanvidence.git",
    "commit": "",                      # pin a sha for reproducibility; empty = HEAD
    "drive_data_root": "/content/drive/MyDrive/BraTSGLI/data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "local_data_root": "/content/brats",
    "drive_zip": "",                   # OR: MyDrive path to ONE .zip of the training folder (overrides drive_data_root reads)
    "out_dir_drive": "/content/drive/MyDrive/scanvidence_runs/full-b0-seed17-colab",
    "seed": 17, "epochs": 150, "lr": 3e-4, "weight_decay": 1e-5,
    "accum": 2, "patch": 96, "foreground_prob": 0.7, "val_every": 5,
    "use_amp": True,   # Colab GPUs have tensor cores. KEEP THIS FLAG IDENTICAL for B1/H0 runs.
    "widths": (16, 32, 64, 128), "num_classes": 4, "num_workers": 4,
}

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q nibabel
!git clone -q {CONFIG["repo_url"]} /content/Scanvidence
import os, sys
os.chdir("/content/Scanvidence")
if CONFIG["commit"]:
    !git checkout -q {CONFIG["commit"]}
for p in ("/content/Scanvidence/src", "/content/Scanvidence"):
    if p not in sys.path:
        sys.path.insert(0, p)
COMMIT = !git rev-parse --short HEAD
print("commit:", COMMIT[0])
assert os.path.isdir(CONFIG["drive_data_root"]), "point CONFIG['drive_data_root'] at the extracted training folder"

In [ ]:
import json
cands = [
    "/content/Scanvidence/data/manifests/split_v1.json",
    "/content/drive/MyDrive/Scanvidence/data/manifests/split_v1.json",
    "/content/drive/MyDrive/split_v1.json",
]
split_path = next((p for p in cands if os.path.exists(p)), None)
assert split_path, "commit split_v1.json to the repo or upload it to Drive"
split = json.load(open(split_path))
train_ids, val_ids = split["train"], split["val"]
sha_path = split_path + ".sha256"
split_hash = open(sha_path).read().strip() if os.path.exists(sha_path) else ""
assert not (set(train_ids) & set(val_ids) & set(split["test"]) & set(split["calibration"]))
print(f"train {len(train_ids)} | val {len(val_ids)} | hash {split_hash[:12]}...")
# Governance: this run touches train+val ONLY. test/cal stay locked.

In [ ]:
import shutil, zipfile
from pathlib import Path

src_root, dst_root = Path(CONFIG["drive_data_root"]), Path(CONFIG["local_data_root"])
need = sorted(set(train_ids) | set(val_ids))

def already_local(c):
    return (dst_root / c / f"{c}-seg.nii.gz").exists()

if CONFIG.get("drive_zip"):
    zp = Path(CONFIG["drive_zip"])
    assert zp.exists(), f"missing zip on Drive: {zp}"
    if all(already_local(c) for c in need):
        print("cases already extracted locally; skipping unzip")
    else:
        print(f"unzipping {zp.name} ({zp.stat().st_size / 1e9:.1f} GB) -> {dst_root}")
        dst_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(dst_root)
        # tolerate wrapper folders inside the zip (e.g. doubled TrainingData layout)
        for _ in range(4):
            entries = list(dst_root.iterdir())
            if len(entries) != 1 or not entries[0].is_dir() or entries[0].name.startswith("BraTS-"):
                break
            inner = entries[0]
            for child in inner.iterdir():
                shutil.move(str(child), str(dst_root / child.name))
            inner.rmdir()
else:
    missing = [c for c in need if not already_local(c)]
    print(f"copying {len(missing)} cases ({len(need) - len(missing)} already local)")
    for i, c in enumerate(missing):
        src_dir, dst_dir = src_root / c, dst_root / c
        dst_dir.mkdir(parents=True, exist_ok=True)
        files = list(src_dir.glob("*.nii.gz")) if src_dir.is_dir() else list(src_root.glob(f"{c}-*.nii.gz"))
        for f in files:
            shutil.copy2(f, dst_dir / f.name)
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(missing)}")

assert all(already_local(c) for c in need), "local data incomplete - check drive_zip / drive_data_root"
print("local copy complete")


In [ ]:
import numpy as np
import nibabel as nib
import torch
from torch.utils.data import Dataset, DataLoader

seed = CONFIG["seed"]
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
device = torch.device("cuda")
print(torch.__version__, "|", torch.cuda.get_device_name(0))

MODS = ("t1n", "t1c", "t2w", "t2f")
ROOT = Path(CONFIG["local_data_root"])

def normalize(v):
    v = np.asarray(v, dtype=np.float32); m = v > 0
    out = np.zeros_like(v)
    if m.any(): out[m] = (v[m] - v[m].mean()) / (v[m].std() + 1e-8)
    return out

def load_case(cid):
    d = ROOT / cid
    vol = np.stack([normalize(nib.load(str(d / f"{cid}-{m}.nii.gz")).get_fdata()) for m in MODS]).astype(np.float32)
    seg = nib.load(str(d / f"{cid}-seg.nii.gz")).get_fdata().astype(np.int64)
    assert set(np.unique(seg).tolist()) <= {0, 1, 2, 3}, f"label drift in {cid}"
    return vol, seg

def _pad(vol, seg, p):
    pad = [(0, max(0, p - s)) for s in vol.shape[1:]]
    if any(q for _, q in pad):
        vol = np.pad(vol, ((0, 0), *pad)); seg = np.pad(seg, pad)
    return vol, seg

def regions_np(L):
    return {"ET": L == 3, "TC": (L == 1) | (L == 3), "WT": (L >= 1) & (L <= 3)}

def dice_empty_rule(a, b):
    a, b = a.astype(bool), b.astype(bool)
    if not a.any() and not b.any(): return 1.0
    if not a.any() or not b.any(): return 0.0
    return float(2 * (a & b).sum() / (a.sum() + b.sum()))

class PatchDataset(Dataset):
    def __init__(self, ids, epoch):
        self.ids, self.epoch = ids, epoch

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        rng = np.random.default_rng([seed, self.epoch, i])
        vol, seg = load_case(self.ids[i])
        p = CONFIG["patch"]; vol, seg = _pad(vol, seg, p)
        idx = np.argwhere(seg > 0)
        c = idx[rng.integers(len(idx))] if (len(idx) and rng.random() < CONFIG["foreground_prob"]) \
            else [rng.integers(0, s) for s in vol.shape[1:]]
        st = [max(0, min(int(x) - p // 2, vol.shape[1 + a] - p)) for a, x in enumerate(c)]
        sl = (slice(None), *[slice(s, s + p) for s in st])
        return torch.from_numpy(vol[sl]), torch.from_numpy(seg[sl])

def val_crop(vol, seg, p):
    vol, seg = _pad(vol, seg, p)
    idx = np.argwhere(seg > 0)
    c = idx.mean(0) if len(idx) else np.array(vol.shape[1:]) / 2
    st = [max(0, min(int(x) - p // 2, vol.shape[1 + a] - p)) for a, x in enumerate(c)]
    sl = (slice(None), *[slice(s, s + p) for s in st])
    return vol[sl], seg[sl]

In [ ]:
from pathlib import Path

from scanvidence.models.backbone import SegResNetB0
model = SegResNetB0(in_channels=4, num_classes=4, widths=CONFIG["widths"], dropout=0.0).to(device)
n_params = sum(p.numel() for p in model.parameters()); assert n_params == 1599420, n_params
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
steps_per_epoch = len(train_ids) // CONFIG["accum"]
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps_per_epoch * CONFIG["epochs"])
scaler = torch.amp.GradScaler("cuda", enabled=CONFIG["use_amp"])

def loss_fn(logits, target):
    ce = torch.nn.functional.cross_entropy(logits, target)
    p = logits.softmax(1); d = 0.0
    for c in (1, 2, 3):
        pc, gc = p[:, c], (target == c).float()
        d = d + (2 * (pc * gc).sum() + 1e-5) / (pc.sum() + gc.sum() + 1e-5)
    return (1 - d / 3) + 0.5 * ce

out_drive = Path(CONFIG["out_dir_drive"]); out_drive.mkdir(parents=True, exist_ok=True)
last_path, best_path = out_drive / "last.pt", out_drive / "best.pt"
hist_path = out_drive / "history.json"
start_epoch, best = 0, -1.0
if last_path.exists():
    ck = torch.load(last_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["state_dict"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, best = ck["epoch"] + 1, ck["best_val_dice"]
    print(f"resuming at epoch {start_epoch}, best {best:.4f}")
history = json.loads(hist_path.read_text()) if hist_path.exists() else []

def save_ckpts(epoch, tag_best):
    payload = dict(
        state_dict=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
        scaler=scaler.state_dict(), epoch=epoch, best_val_dice=best, params=n_params,
        config={**CONFIG, "widths": list(CONFIG["widths"])}, split_hash=split_hash,
        gpu=torch.cuda.get_device_name(0), torch=torch.__version__, commit=COMMIT[0],
    )
    torch.save(payload, last_path)
    if tag_best:
        torch.save(payload, best_path)

In [ ]:
import time

try:
    for epoch in range(start_epoch, CONFIG["epochs"]):
        model.train()
        ids = train_ids[:]; np.random.default_rng(seed + epoch).shuffle(ids)
        dl = DataLoader(PatchDataset(ids, epoch), batch_size=1, num_workers=CONFIG["num_workers"],
                        prefetch_factor=2, pin_memory=True)
        losses, t0 = [], time.time()
        for i, (x, y) in enumerate(dl):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=CONFIG["use_amp"]):
                o = model(x)
                if isinstance(o, (tuple, list)): o = o[0]
                loss = loss_fn(o, y) / CONFIG["accum"]
            scaler.scale(loss).backward()
            if (i + 1) % CONFIG["accum"] == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            losses.append(float(loss) * CONFIG["accum"])
        val_score = -1.0
        if (epoch + 1) % CONFIG["val_every"] == 0 or epoch == CONFIG["epochs"] - 1:
            model.eval(); scores = []
            with torch.no_grad():
                for cid in val_ids:
                    vol, seg = load_case(cid); vx, vy = val_crop(vol, seg, CONFIG["patch"])
                    vx = torch.from_numpy(vx)[None].to(device)
                    vy = torch.from_numpy(vy)[None].to(device)
                    with torch.amp.autocast("cuda", enabled=CONFIG["use_amp"]):
                        o = model(vx)
                        if isinstance(o, (tuple, list)): o = o[0]
                    pr = regions_np(o.argmax(1)[0].cpu().numpy()); gr = regions_np(seg)
                    scores.append(np.mean([dice_empty_rule(pr[r], gr[r]) for r in ("ET", "TC", "WT")]))
            val_score = float(np.mean(scores))
        tag = val_score > best
        if tag: best = val_score
        save_ckpts(epoch, tag)
        history.append(dict(epoch=epoch, train_loss=round(float(np.mean(losses)), 4),
                            val_dice=round(val_score, 4), dt=round(time.time() - t0, 1)))
        print(f"[epoch {epoch+1}/{CONFIG['epochs']}] loss {history[-1]['train_loss']:.4f} "
              f"val {val_score:.4f} best {best:.4f}{' *' if tag else ''} ({history[-1]['dt']}s)", flush=True)
except KeyboardInterrupt:
    print("interrupted - last.pt already on Drive; re-run cells 6+7 to resume")

In [ ]:
import json
(out_drive / "history.json").write_text(json.dumps(history, indent=2))
(out_drive / "run.json").write_text(json.dumps(dict(
    model="B0 (SegResNetB0)", params=n_params, seed=seed, split_hash=split_hash,
    precision="amp" if CONFIG["use_amp"] else "fp32", best_val_dice=best,
    gpu=torch.cuda.get_device_name(0), torch=torch.__version__, commit=COMMIT[0],
    config={**CONFIG, "widths": list(CONFIG["widths"])}), indent=2))
print(f"done. best val Dice {best:.4f} -> {best_path}")
print("Next: copy best.pt to your workstation and point frozen_validation_viewer at it")
print("(CONFIG['ckpt_b0'] = that path) for the full-volume frozen-val evaluation.")